# Kenya Land Readiness Atlas: baseline notebook

Run top to bottom on Colab (CPU runtime is enough, no GPU needed).

What it does: pulls Sentinel-2 features and WorldCover labels for 4 contrasting regions through Earth Engine, caches them to Drive, builds buffered spatial-block folds, trains a shallow decision tree and a LightGBM baseline, evaluates with spatial CV and leave-one-region-out, and reports a confusion matrix, prevalence-weighted accuracy and calibration.

Before running: set PROJECT in the config cell to your Google Cloud project that is registered for Earth Engine.

## 1. Install

In [56]:
!pip -q install -U earthengine-api lightgbm

## 2. Config

In [57]:
PROJECT = "propertysatellite"   # <-- set this

YEAR     = 2021      # WorldCover v200 is a 2021 map, so use 2021 imagery
PER_CLASS = 400      # target samples per class per region (stratified)
CLOUD_THRESH = 0.6   # Cloud Score+ cs_cdf threshold
BLOCK_DEG    = 0.05  # spatial block size in degrees (~5.5 km)
N_FOLDS = 5
SEED    = 42
USE_DRIVE = True
OUT_DIR   = "/content/drive/MyDrive/land_atlas_baseline" if USE_DRIVE else "/content/land_atlas_baseline"
EDA_DIR   = OUT_DIR + "/eda"           # EDA outputs mirror on Drive

CLASSES = ["built_up", "cropland", "trees", "grass_shrub", "water", "bare"]

# ── 10 counties across 5 climate zones ──────────────────────────────────────
# [min_lon, min_lat, max_lon, max_lat]
REGIONS = {
    # Highland zone (Central Highlands — cool, wet, high NDVI)
    "highland":   [36.30, -0.60, 36.80, -0.10],
    "muranga":    [36.90, -0.90, 37.40, -0.40],
    "nyandarua":  [36.20, -0.80, 36.70, -0.20],
    # Coastal zone (humid, tropical, mangrove)
    "coast":      [39.40, -3.90, 39.90, -3.40],
    "kwale":      [39.10, -4.40, 39.65, -3.80],
    "lamu":       [40.70, -2.40, 41.30, -1.80],
    # ASAL zone (arid & semi-arid, sparse vegetation)
    "arid":       [37.50,  1.50, 38.00,  2.00],
    "turkana":    [35.20,  3.20, 35.80,  3.80],
    "wajir":      [40.20,  1.60, 40.80,  2.20],
    # Lake zone (Lake Victoria basin, sub-humid, wetland)
    "lake_basin": [34.60, -0.30, 35.10,  0.20],
    "siaya":      [34.00, -0.30, 34.60,  0.30],
    # Rift Valley zone (savanna, pyroclastic soils)
    "nakuru":     [35.80, -0.60, 36.30, -0.10],
    "kajiado":    [36.60, -2.00, 37.20, -1.40],
}

CLIMATE_ZONES = {
    "highland": "highland_zone", "muranga": "highland_zone", "nyandarua": "highland_zone",
    "coast":    "coastal_zone",  "kwale":   "coastal_zone",  "lamu":      "coastal_zone",
    "arid":     "asal_zone",     "turkana": "asal_zone",     "wajir":     "asal_zone",
    "lake_basin":"lake_zone",    "siaya":   "lake_zone",
    "nakuru":   "rift_zone",     "kajiado": "rift_zone",
}


In [62]:
import os

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/Satellite'):
    !git clone https://github.com/VAL-Jerono/Satellite.git /content/Satellite

%cd /content/Satellite
!git pull

!pip install -q -r requirements.txt opencv-python-headless

!python -m data.pipeline.patch_export --project propertysatellite --workers 12


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Satellite


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.04 KiB | 530.00 KiB/s, done.
From https://github.com/VAL-Jerono/Satellite
   1149506..5c351ac  main       -> origin/main
Updating 1149506..5c351ac
Fast-forward
 data/pipeline/run_pipeline.py | 28 ++++++++++++++++++++++++++++
 1 file changed, 28 insertions(+)


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

...

Drive patch sync complete.
Patches to download: 0 / 8356


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Manifest saved: /content/Satellite/data/patches/manifest.csv (8356 patches indexed)


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

Backed up patches and manifest to Drive: /content/drive/MyDrive/land_atlas_baseline/patches


In [63]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/Satellite
!git pull origin main

import ee
PROJECT = "propertysatellite"  # <-- Replace with your GCP project ID

# Authenticate Earth Engine INTERACTIVELY in the notebook cell:
try:
    ee.Initialize(project=PROJECT)
    print("Earth Engine initialized successfully!")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    print("Earth Engine authenticated & initialized!")


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Satellite
From https://github.com/VAL-Jerono/Satellite
 * branch            main       -> FETCH_HEAD
Already up to date.


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Earth Engine initialized successfully!


## 3. Earth Engine and Drive

In [64]:
import os, ee
assert PROJECT not in ("your-gcp-project-id", "", None), "Set PROJECT in the config cell first."
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine ready")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)
CACHE = f"{OUT_DIR}/samples_{YEAR}_n{PER_CLASS}.csv"
print("Cache:", CACHE, "| exists:", os.path.exists(CACHE))

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Earth Engine ready


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cache: /content/drive/MyDrive/land_atlas_baseline/samples_2021_n400.csv | exists: True


## 4. Data: Sentinel-2 features and WorldCover labels

Labels are WorldCover 2021 remapped to 6 classes (wetland to water, mangrove to trees, snow and moss dropped). This measures agreement with WorldCover, not ground truth. The independent hand-labelled test set comes later.

In [65]:
# ── Data pull via batch-safe orchestrator ─────────────────────────────────────
# Runs IN-PROCESS (same kernel where pip installed earthengine-api).
# Reads ledger.json — skips counties already marked done.  Safe to re-run.
#
# Tweak these to control scope:
BATCH_SIZE = 2          # counties per batch (lower = safer on free Colab tier)
ONLY_COUNTIES = None    # None = all 13 counties
# ONLY_COUNTIES = ["highland", "muranga"]   # ← uncomment to test 2 counties

import sys, importlib

# Ensure repo root is on path
if "/content/Satellite" not in sys.path:
    sys.path.insert(0, "/content/Satellite")

# ── Reload ALL pipeline modules so git pull changes take effect ───────────────
# (Python caches modules in sys.modules; reload forces re-read from disk)
import data.pipeline.ee_pull        as _ee_pull
import data.pipeline.patch_export   as _patch_export
import data.pipeline.spatial_splits as _spatial_splits
import data.pipeline.ledger         as _ledger
import data.pipeline.run_pipeline   as _rp

for _mod in [_ee_pull, _patch_export, _spatial_splits, _ledger, _rp]:
    importlib.reload(_mod)

print("All pipeline modules reloaded from disk.")

# ── Run the pipeline ──────────────────────────────────────────────────────────
_rp.run_pipeline(
    project    = PROJECT,
    batch_size = BATCH_SIZE,
    counties   = ONLY_COUNTIES,
)

# ── Load the merged master CSV ────────────────────────────────────────────────
import pandas as pd, numpy as np
from pathlib import Path

local_cache = Path("data/raw") / f"samples_{YEAR}_n{PER_CLASS}.csv"
drive_cache = Path(f"{OUT_DIR}/samples_{YEAR}_n{PER_CLASS}.csv")

if local_cache.exists():
    df = pd.read_csv(local_cache)
    print(f"Loaded local cache: {df.shape}")
elif drive_cache.exists():
    df = pd.read_csv(drive_cache)
    print(f"Loaded Drive cache: {df.shape}")
else:
    raise FileNotFoundError(
        "Master CSV not found — run_pipeline must complete the merge_csv stage first.\n"
        "Tip: check the ledger above — any county with ee_pull:I means it failed."
    )

BANDS    = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
IDX      = ["NDVI", "NDWI", "MNDWI", "NDBI"]
EXTRA    = ["NDVI_p10", "NDVI_p90", "NDVI_stdDev", "elev", "slope"]
FEATURES = BANDS + IDX + EXTRA
K        = len(CLASSES)   # 6 land-cover classes

df = df.dropna(subset=FEATURES + ["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
if "climate_zone" not in df.columns and "region" in df.columns:
    df["climate_zone"] = df["region"].map(CLIMATE_ZONES).fillna("unknown")

N = len(df)
print(f"Final Dataset Shape: {df.shape}")
print(f"Counties present ({df['region'].nunique()}): {sorted(df['region'].unique())}")
display(pd.crosstab(df["region"], df["label"].map(dict(enumerate(CLASSES)))))


All pipeline modules reloaded from disk.
[Pipeline] Local chunk cache already warm (208 files)
=== Pipeline Ledger ===
  highland     ee_pull:D  patch_export:D  splits:P
  muranga      ee_pull:D  patch_export:I  splits:P
  nyandarua    ee_pull:D  patch_export:I  splits:P
  coast        ee_pull:D  patch_export:I  splits:P
  kwale        ee_pull:D  patch_export:I  splits:P
  lamu         ee_pull:D  patch_export:I  splits:P
  arid         ee_pull:D  patch_export:I  splits:P
  turkana      ee_pull:D  patch_export:I  splits:P
  wajir        ee_pull:D  patch_export:I  splits:P
  lake_basin   ee_pull:D  patch_export:I  splits:P
  siaya        ee_pull:D  patch_export:I  splits:P
  nakuru       ee_pull:D  patch_export:I  splits:P
  kajiado      ee_pull:D  patch_export:I  splits:P
  [global] band_stats=stale  splits=done

[Pipeline] All counties done for stage 'ee_pull' — skipping

[Pipeline] === Stage: patch_export | 12 counties remaining ===

[Pipeline] Batch: ['muranga', 'nyandarua']
  [PATCH

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

  [PATCH] Need to download: 268 / 2388


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

  [PATCH] 50/268 downloaded


...

  [PATCH] 1050/2365 downloaded


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

  [PATCH] 1100/2365 downloaded


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mo

KeyboardInterrupt: 

## 5. Spatial blocks and buffered folds

Points are grouped into 0.05 degree blocks. Whole blocks go to the test fold, and any training point in a block touching a test block is dropped (buffer), so neighbouring pixels cannot leak across the split.

In [ ]:
from sklearn.model_selection import GroupKFold

df["bx"] = np.floor(df["lon"] / BLOCK_DEG).astype(int)
df["by"] = np.floor(df["lat"] / BLOCK_DEG).astype(int)
df["block"] = df["region"] + "_" + df["bx"].astype(str) + "_" + df["by"].astype(str)
print("Blocks:", df["block"].nunique())

reg, bx, by = df["region"].values, df["bx"].values, df["by"].values

def buffered_train(train_idx, test_idx):
    test_blocks = set(zip(reg[test_idx], bx[test_idx], by[test_idx]))
    keep = []
    for i in train_idx:
        near = any((reg[i], bx[i] + dx, by[i] + dy) in test_blocks
                   for dx in (-1, 0, 1) for dy in (-1, 0, 1))
        if not near:
            keep.append(i)
    return np.array(keep)

gkf = GroupKFold(n_splits=N_FOLDS)
FOLDS = []
for tr, te in gkf.split(df, df["label"], groups=df["block"]):
    tr_b = buffered_train(tr, te)
    FOLDS.append((tr_b, te))
    print(f"fold: train {len(tr)} -> {len(tr_b)} after buffer | test {len(te)}")

## 6. Models and metrics helpers

In [ ]:
import lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix

X = df[FEATURES].values
y = df["label"].values
w = df["weight"].values
N = len(df)

LGB_PARAMS = dict(objective="multiclass", n_estimators=300, learning_rate=0.05,
                  num_leaves=31, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
RULE_FEATS = [FEATURES.index(c) for c in ["NDVI", "NDWI", "MNDWI", "NDBI", "elev"]]

def make_model(name):
    if name == "tree_rule":
        return DecisionTreeClassifier(max_depth=4, random_state=SEED)
    return lgb.LGBMClassifier(**LGB_PARAMS)

def cols(name):
    return RULE_FEATS if name == "tree_rule" else list(range(len(FEATURES)))

def fit_predict(name, tr, te):
    m = make_model(name)
    c = cols(name)
    m.fit(X[tr][:, c], y[tr])
    p = np.zeros((len(te), K))
    p[:, m.classes_] = m.predict_proba(X[te][:, c])
    return m, p

def metrics(y_true, proba, weights):
    pred = proba.argmax(1)
    return dict(acc=float((pred == y_true).mean()),
                weighted_acc=float(np.average(pred == y_true, weights=weights)),
                macro_f1=float(f1_score(y_true, pred, average="macro")))

def ece(proba, y_true, bins=10):
    conf, pred = proba.max(1), proba.argmax(1)
    ok = (pred == y_true).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    out, tot = 0.0, len(y_true)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.sum() / tot * abs(ok[m].mean() - conf[m].mean())
    return float(out)

## 7. Spatial block CV (buffered)

In [ ]:
MODELS = ["tree_rule", "lgbm"]
oof = {m: np.full((N, K), np.nan) for m in MODELS}

for name in MODELS:
    for i, (tr, te) in enumerate(FOLDS):
        _, p = fit_predict(name, tr, te)
        oof[name][te] = p
    print("done", name)

rows = []
for name in MODELS:
    r = metrics(y, oof[name], w)
    r["ece"] = ece(oof[name], y)
    r["model"] = name
    rows.append(r)
cv_table = pd.DataFrame(rows).set_index("model").round(4)
display(cv_table)

## 8. Leave-one-region-out

The harder test: train on 3 regions, test on the 4th. Expect lower numbers than block CV. That gap is the honest generalisation story.

In [ ]:
# Leave-One-Region-Out (LORO) — works with any subset of counties present in df
# Skips held-out counties that have no data yet (safe during incremental pull)
loro_rows = []
counties_in_df = set(df["region"].unique()) if "region" in df.columns else set()

for held in REGIONS:
    te = np.where(df["region"].values == held)[0]
    if len(te) == 0:
        print(f"  SKIP LORO {held}: no data in df yet (run pipeline to pull this county)")
        continue
    tr = np.where(df["region"].values != held)[0]
    if len(tr) == 0:
        print(f"  SKIP LORO {held}: training set would be empty")
        continue
    for name in MODELS:
        _, p = fit_predict(name, tr, te)
        r = metrics(y[te], p, w[te])
        r.update(held_out=held, model=name)
        loro_rows.append(r)

if loro_rows:
    loro = pd.DataFrame(loro_rows).set_index(["held_out", "model"]).round(4)
    display(loro)
    print(f"\nLORO mean ({len(counties_in_df)} counties present):")
    display(loro.groupby("model").mean().round(4).rename(columns=lambda c: c + " (mean)"))
else:
    print("No LORO results yet — pull at least 2 counties first.")


## 9. Confusion matrix, per-region error, calibration

In [ ]:
import matplotlib.pyplot as plt

pred = oof["lgbm"].argmax(1)
cm = confusion_matrix(y, pred, labels=range(K), normalize="true")

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

im = ax[0].imshow(cm, vmin=0, vmax=1, cmap="Blues")
ax[0].set_xticks(range(K)); ax[0].set_xticklabels(CLASSES, rotation=45, ha="right")
ax[0].set_yticks(range(K)); ax[0].set_yticklabels(CLASSES)
for i in range(K):
    for j in range(K):
        ax[0].text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                   color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
ax[0].set_title("LightGBM confusion matrix (row-normalised, block CV)")
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("WorldCover label")

reg_acc = (pd.DataFrame({"region": df["region"], "ok": pred == y, "w": w})
           .groupby("region").apply(lambda g: np.average(g["ok"], weights=g["w"])))
reg_acc.plot.bar(ax=ax[1], color="steelblue")
ax[1].set_ylim(0, 1); ax[1].set_title("Weighted accuracy by region (block CV)")
ax[1].set_ylabel("accuracy")

conf = oof["lgbm"].max(1); ok = (pred == y).astype(float)
edges = np.linspace(0, 1, 11); xs, ys = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.sum() >= 10:
        xs.append(conf[m].mean()); ys.append(ok[m].mean())
ax[2].plot([0, 1], [0, 1], "k--", label="perfect")
ax[2].plot(xs, ys, "o-", label="LightGBM")
ax[2].set_xlabel("mean confidence"); ax[2].set_ylabel("accuracy")
ax[2].set_title(f"Reliability (ECE = {ece(oof['lgbm'], y):.3f})"); ax[2].legend()

plt.tight_layout(); plt.show()

print("Per-class recall (block CV):")
print(pd.Series(np.diag(cm), index=CLASSES).round(3).to_string())

## 10. Feature importance and final model

In [ ]:
final = lgb.LGBMClassifier(**LGB_PARAMS).fit(X, y)
imp = pd.Series(final.booster_.feature_importance("gain"), index=FEATURES).sort_values()
imp.plot.barh(figsize=(6, 7), title="LightGBM gain importance (all data)")
plt.tight_layout(); plt.show()

## 11. Save artefacts and summary

In [ ]:
import json, joblib

final.booster_.save_model(f"{OUT_DIR}/lgbm_baseline.txt")
joblib.dump(final, f"{OUT_DIR}/lgbm_baseline.joblib")

oof_df = df[["region", "block", "lon", "lat", "label"]].copy()
oof_df["pred"] = oof["lgbm"].argmax(1)
oof_df["conf"] = oof["lgbm"].max(1)
oof_df.to_csv(f"{OUT_DIR}/oof_predictions.csv", index=False)

summary = dict(
    year=YEAR, per_class=PER_CLASS, n_points=int(N), classes=CLASSES, features=FEATURES,
    block_cv=cv_table.reset_index().to_dict("records"),
    leave_one_region_out=loro.reset_index().to_dict("records"),
)
with open(f"{OUT_DIR}/baseline_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to", OUT_DIR)
print("\nBlock CV:")
display(cv_table)
print("Leave-one-region-out mean:")
display(loro.groupby("model").mean().round(4))

## Reading these numbers

All accuracy here is agreement with WorldCover, which is itself a model. Weighted accuracy reweights the class-balanced sample by each class's share of the region, so rare classes like built-up do not inflate the headline number. Before you quote any figure, add the independent test: hand-label 300 to 500 stratified random points from high-resolution basemaps and score the model on those with a confusion matrix and area-weighted accuracy (Olofsson et al., 2014).

Next steps from here: swap LightGBM for a ResNet18 or EfficientNet-B0 fine-tune on 64x64 patches, track runs in MLflow, then export the best model to ONNX for CPU serving.

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/Satellite'):
    !git clone https://github.com/VAL-Jerono/Satellite.git /content/Satellite

%cd /content/Satellite
!git pull

!pip install -q -r requirements.txt opencv-python-headless

!python -m data.pipeline.patch_export --project propertysatellite --workers 12


In [ ]:
%cd /content/Satellite
!python -m data.pipeline.spatial_splits


In [ ]:
import os

# Mount Drive to ensure access to your saved sample CSV
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/Satellite'):
    !git clone https://github.com/VAL-Jerono/Satellite.git /content/Satellite

%cd /content/Satellite
!git pull

!pip install -q -r requirements.txt opencv-python-headless


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/Satellite
!git pull origin main

import ee
PROJECT = "propertysatellite"  # <-- Replace with your GCP project ID

# Authenticate Earth Engine INTERACTIVELY in the notebook cell:
try:
    ee.Initialize(project=PROJECT)
    print("Earth Engine initialized successfully!")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    print("Earth Engine authenticated & initialized!")


In [ ]:
# Pull latest Drive backup code & increase workers to 16 for faster download
!git pull origin main

# 1. Export patch tiles (instantly finds your cached samples CSV in Google Drive)
!python -m data.pipeline.patch_export --project $PROJECT --workers 8

# 2. Build spatial block splits
!python -m data.pipeline.spatial_splits --project $PROJECT



In [ ]:
!git pull origin main
!python -m training.train

## 12. Dataset Band Normalization & Exploratory Data Analysis (EDA)

Computes exact 10-band dataset-wide statistics across all 8,356 Sentinel-2 patches to refine `_BAND_MEAN` and `_BAND_STD` vectors, and runs EDA to generate class distribution summaries and spectral signature curves.

In [ ]:
import os, shutil
from pathlib import Path

# ── 1. Compute band normalization statistics ──────────────────────────────────
!python -m data.compute_band_stats

# ── 2. Run full EDA (saves to docs/eda/ locally AND Drive/…/eda/) ────────────
!python -m data.eda

# ── 3. Confirm what was saved and display key figures inline ──────────────────
LOCAL_EDA  = Path("docs/eda")
DRIVE_EDA  = Path(EDA_DIR)          # defined in the Config cell

print("\n===== EDA Output Files =====")
for f in sorted(LOCAL_EDA.rglob("*")):
    if f.is_file():
        drive_copy = DRIVE_EDA / f.relative_to(LOCAL_EDA)
        on_drive = "✓ Drive" if drive_copy.exists() else "✗ Drive"
        print(f"  {on_drive}  |  LOCAL: {f}")

# Display figures inline (Colab / Jupyter)
try:
    from IPython.display import display, Image, Markdown
    report_path = LOCAL_EDA / "eda_report.md"
    if report_path.exists():
        display(Markdown(report_path.read_text()))
    for fig_name in ["spectral_profiles.png", "spectral_indices_distribution.png",
                     "county_class_heatmap.png"]:
        fig_path = LOCAL_EDA / "figures" / fig_name
        if fig_path.exists():
            print(f"\n--- {fig_name} ---")
            display(Image(str(fig_path)))
except Exception as e:
    print(f"Inline display skipped: {e}")


## 13. Fine-Tuned PyTorch CNN (Spatial CV & LORO Fine-Tuning)

Trains the fine-tuned Multi-Spectral Sentinel-2 CNN (EfficientNet-B0 / ResNet-18 with Spectral Channel Attention) across 5 spatial block folds and Leave-One-Region-Out (LORO) splits. Exports the best checkpoint to ONNX (`models/land_cover.onnx`).

In [ ]:
!pip install -q onnx onnxscript
!git pull origin main
!python -m training.train --epochs 30

In [ ]:
# ==============================================================================
# KENYA LAND READINESS ATLAS — STEP 1: FULL CNN TRAINING & ONNX EXPORT
# ==============================================================================
import os
import sys
import json
import torch
from pathlib import Path

# 1. Environment & Path Setup
ROOT_DIR = Path.cwd()
if (ROOT_DIR / "Satellite").exists():
    ROOT_DIR = ROOT_DIR / "Satellite"
sys.path.append(str(ROOT_DIR))
os.chdir(ROOT_DIR)

print(f"Working Directory: {ROOT_DIR}")
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name:   {torch.cuda.get_device_name(0)}")

# 2. Check / Generate Spatial Block Folds & LORO Splits
from data.pipeline.spatial_splits import build_splits

splits_dir = ROOT_DIR / "data" / "splits"
if not (splits_dir / "fold_0_train.csv").exists():
    print("Generating 5-fold buffered spatial block splits + LORO splits...")
    build_splits()

# 3. Execute Multi-Fold Training & LORO Evaluation
from training.train import train_fold, build_model, export_onnx
import yaml

cfg = yaml.safe_load((ROOT_DIR / "configs" / "config.yaml").read_text())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 25

print("\n" + "="*60)
print("STARTING MODEL TRAINING: 5 SPATIAL BLOCK FOLDS + LORO ARID")
print("="*60)

results = []

# A. Train 5 Spatial Folds
for fold in range(5):
    tr_csv = splits_dir / f"fold_{fold}_train.csv"
    va_csv = splits_dir / f"fold_{fold}_val.csv"
    print(f"\n---> Training Spatial Fold {fold}/4")
    ckpt, f1, _ = train_fold(
        fold_id=str(fold),
        train_csv=tr_csv,
        val_csv=va_csv,
        device=device,
        run_name=f"spatial_fold_{fold}",
        epochs=epochs
    )
    results.append({"split": f"fold_{fold}", "macro_f1": f1, "ckpt": str(ckpt)})

# B. Train Leave-One-Region-Out (LORO Arid - Northern Out-of-Domain Benchmark)
loro_region = "arid"
loro_tr = splits_dir / f"loro_{loro_region}_train.csv"
loro_va = splits_dir / f"loro_{loro_region}_val.csv"
print(f"\n---> Training LORO Split (Held-out Region: {loro_region.upper()})")
ckpt_loro, f1_loro, _ = train_fold(
    fold_id=f"loro_{loro_region}",
    train_csv=loro_tr,
    val_csv=loro_va,
    device=device,
    run_name=f"loro_{loro_region}",
    epochs=epochs
)
results.append({"split": f"loro_{loro_region}", "macro_f1": f1_loro, "ckpt": str(ckpt_loro)})

# 4. Export Best Checkpoint to ONNX Format
best_run = max(results, key=lambda r: r["macro_f1"])
model_dir = ROOT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)
onnx_path = model_dir / "land_cover.onnx"

print("\n" + "="*60)
print(f"EXPORTING TOP MODEL ({best_run['split']}) TO ONNX...")
print("="*60)

best_model = build_model()
checkpoint = torch.load(best_run["ckpt"], map_location="cpu")
best_model.load_state_dict(checkpoint["model_state"])

export_onnx(
    model=best_model,
    out_path=onnx_path,
    patch_size=cfg["training"]["patch_size"],
    in_channels=cfg["model"]["in_channels"]
)

# 5. Benchmark Verification Assertion
print("\n" + "="*60)
print("FINAL BENCHMARK EVALUATION SUMMARY")
print("="*60)

for r in results:
    print(f"  {r['split']:<15} | Macro-F1: {r['macro_f1']:.4f}")

lgb_baseline_loro = 0.4154
print(f"\nLightGBM LORO Baseline: {lgb_baseline_loro:.4f}")
print(f"CNN + Attention LORO:   {f1_loro:.4f}")

if f1_loro > 0.60:
    print("✅ SUCCESS: CNN + Spectral Attention exceeds target benchmark (> 0.60 Macro-F1)!")
elif f1_loro > lgb_baseline_loro:
    print("⚠️ PROGRESS: CNN outperforms LightGBM baseline but did not reach 0.60. Consider more epochs.")
else:
    print("❌ WARNING: Performance below baseline. Check learning rate schedule and data augmentations.")

# 6. Verify ONNX Model Load via ONNX Runtime
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
dummy_input = np.random.randn(1, 10, 64, 64).astype(np.float32)
output = sess.run(None, {"sentinel2_patch": dummy_input})[0]

print(f"\nONNX Inference Verification Successful!")
print(f"Input Shape:  (1, 10, 64, 64)")
print(f"Output Logits Shape: {output.shape} -> Classes: {cfg['model']['num_classes']}")


In [ ]:
# 1. Ensure your trained ONNX model is in place:
cp models/land_cover.onnx models/land_cover.onnx

# 2. Launch the full multi-container stack:
docker compose up --build

# Services launched:
# - FastAPI API:   http://localhost:8000/docs
# - Streamlit Map: http://localhost:8501
# - MLflow Server: http://localhost:5000

# 3. Simulate Seasonal Feature Drift Report (Evidently AI):
python -m monitoring.drift --simulate
